# 🧬 Factor Regression Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/04_factor_analysis.ipynb)

CAPM, Fama-French 3, Carhart 4, FF5, FF5+Momentum — factor data auto-downloaded free from the Ken French library, with Newey-West t-stats and rolling exposures. Is your fund's "alpha" just factor exposure?

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Settings {display-mode: "form"}
ticker = "BRK-B"        #@param {type:"string"}
model = "ff5_mom"       #@param ["capm", "ff3", "carhart", "ff5", "ff5_mom"]
start_date = "2000-01-01"  #@param {type:"date"}
rolling_window_months = 36 #@param {type:"number"}
compare_tickers = "SPY, QQQ, VTV, MTUM, USMV"  #@param {type:"string"}
SMOKE = False

In [ ]:
from portlab.data import get_prices
from portlab.data.factors import get_ff_factors
from portlab.factor import compare_funds, factor_regression, rolling_exposures
from portlab.returns import simple_returns
from portlab import plots

t = ticker.strip().upper()
prices = get_prices([t], start_date, interval="1d")
monthly = (1 + simple_returns(prices[t])).resample("ME").prod() - 1
factors = get_ff_factors(model, "monthly", start=start_date)

tbl = factor_regression(monthly, factors, model=model)
print(f"R² = {tbl.attrs['r_squared']:.3f}   adj R² = {tbl.attrs['r_squared_adj']:.3f}   n = {tbl.attrs['n_obs']}")
tbl.style.format({"loading": "{:.4f}", "t_stat": "{:.2f}", "p_value": "{:.4f}"}, na_rep="—")

In [ ]:
roll = rolling_exposures(monthly, factors, model="ff3", window=rolling_window_months)
plots.rolling_chart(roll.drop(columns="alpha"), f"{t} — Rolling FF3 Exposures ({rolling_window_months}m)", yformat=".2f").show()

In [ ]:
comp = [c.strip().upper() for c in compare_tickers.split(",") if c.strip()]
cprices = get_prices(comp, start_date)
crets = (1 + simple_returns(cprices)).resample("ME").prod() - 1
compare_funds(crets, factors, model="ff3").style.format("{:.3f}")

In [ ]:
# Factor performance attribution — how much return came from each exposure?
from portlab.factor import attribution
contrib = attribution(monthly, factors, model="ff3")
print("Cumulative contribution by source:")
display(contrib.attrs["total"].to_frame("total return").style.format("{:.2%}"))
plots.rolling_chart(contrib.cumsum(), f"{t} — Cumulative Factor Attribution").show()

In [ ]:
#@title Match factor exposure — replicate a fund with other assets {display-mode: "form"}
target_ticker = "ARKK"      #@param {type:"string"}
building_blocks = "QQQ, IWM, XBI, SPY"  #@param {type:"string"}
from portlab.factor import match_exposure
tt = target_ticker.strip().upper()
blocks = [c.strip().upper() for c in building_blocks.split(",") if c.strip()]
px = get_prices([tt] + blocks, start_date)
mr = (1 + simple_returns(px)).resample("ME").prod() - 1
res = match_exposure(mr[tt].dropna(), mr[blocks].dropna(), factors=factors)
print(f"Tracking error: {res['tracking_error']:.2%}/yr")
display(res["weights"].to_frame("replica weight").style.format("{:.1%}"))
res["loadings"].style.format("{:.3f}")